# 02 Baseline Modeling: Interpretable Logistic Regression

This notebook adds a simple, reproducible baseline modeling workflow for the UCI processed Cleveland Heart Disease dataset. It is an educational risk-modeling exercise, not a diagnostic medical tool.

## Scope and Caution

The goal is to compare a transparent logistic regression baseline against a dummy majority-class baseline on a held-out test set. No clinical validity is claimed. Results describe this local dataset split only and should not be used for medical decisions.

## Dataset Citation

Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). Heart Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import (  # noqa: E402
    HEART_DISEASE_COLUMNS,
    binarize_heart_disease_target,
    clean_missing_values,
    load_heart_disease_data,
)
from src.evaluation import (  # noqa: E402
    classification_report_table,
    compute_classification_metrics,
    plot_confusion_matrix,
    plot_logistic_coefficients,
    plot_precision_recall_curve,
    plot_roc_curve,
)
from src.features import (  # noqa: E402
    define_heart_disease_feature_columns,
    feature_column_names,
)
from src.modeling import (  # noqa: E402
    create_dummy_classifier,
    create_logistic_regression_pipeline,
    split_train_test,
)

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

## Load and Prepare Data

In [2]:
def load_modeling_data(path: Path) -> pd.DataFrame:
    """Load local data and create the exploratory binary target."""
    if not path.exists():
        raise FileNotFoundError(
            "Expected data/raw/processed.cleveland.data. Download the UCI Heart Disease files from "
            "https://archive.ics.uci.edu/dataset/45/heart+disease and place the processed Cleveland file there."
        )
    data = clean_missing_values(load_heart_disease_data(path))
    for column in HEART_DISEASE_COLUMNS:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data["target_binary"] = binarize_heart_disease_target(data, "target")["target"]
    return data


df = load_modeling_data(RAW_DATA_PATH)
print(f"Rows: {df.shape[0]}")
print(f"Original columns: {len(HEART_DISEASE_COLUMNS)}")
print(df[["target", "target_binary"]].head().to_string(index=False))

Rows: 303
Original columns: 14
 target  target_binary
      0              0
      2              1
      1              1
      0              0
      0              0


The original `target` column is preserved. `target_binary` maps original `0` to `0` and values greater than `0` to `1` for this educational binary classification exercise.

In [3]:
feature_columns = define_heart_disease_feature_columns()
model_features = feature_column_names(feature_columns)
x = df[model_features].copy()
y = df["target_binary"].copy()

print(f"Feature columns: {model_features}")
print(y.value_counts().sort_index().rename("count").to_frame().to_string())

Feature columns: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
               count
target_binary       
0                164
1                139


## Stratified Train/Test Split

In [4]:
x_train, x_test, y_train, y_test = split_train_test(
    x,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training rows: {len(x_train)}")
print(f"Test rows: {len(x_test)}")
print("\nTraining target distribution")
print(y_train.value_counts(normalize=True).sort_index().round(3).to_string())
print("\nTest target distribution")
print(y_test.value_counts(normalize=True).sort_index().round(3).to_string())

Training rows: 242
Test rows: 61

Training target distribution
target_binary
0    0.541
1    0.459

Test target distribution
target_binary
0    0.541
1    0.459


Preprocessing is part of the scikit-learn pipeline. Imputation, scaling, and one-hot encoding are fitted on training data only, then applied to the held-out test set.

## Fit Baseline Models

In [5]:
dummy_model = create_dummy_classifier(random_state=42)
logistic_model = create_logistic_regression_pipeline(feature_columns)

dummy_model.fit(x_train, y_train)
logistic_model.fit(x_train, y_train)

dummy_predictions = dummy_model.predict(x_test)
dummy_probabilities = dummy_model.predict_proba(x_test)[:, 1]
logistic_predictions = logistic_model.predict(x_test)
logistic_probabilities = logistic_model.predict_proba(x_test)[:, 1]

## Held-Out Test Metrics

In [6]:
dummy_metrics = compute_classification_metrics(
    y_test, dummy_predictions, dummy_probabilities
)
logistic_metrics = compute_classification_metrics(
    y_test, logistic_predictions, logistic_probabilities
)

metrics_table = pd.DataFrame(
    [dummy_metrics, logistic_metrics],
    index=["Dummy majority baseline", "Logistic regression baseline"],
).round(3)
print(metrics_table.to_string())

                              accuracy  precision  recall     f1  roc_auc  average_precision
Dummy majority baseline          0.541      0.000   0.000  0.000    0.500              0.459
Logistic regression baseline     0.869      0.812   0.929  0.867    0.966              0.963


In [7]:
print("Logistic regression classification report")
print(classification_report_table(y_test, logistic_predictions).round(3).to_string())

Logistic regression classification report
              precision  recall  f1-score  support
0                 0.931   0.818     0.871   33.000
1                 0.812   0.929     0.867   28.000
accuracy          0.869   0.869     0.869    0.869
macro avg         0.872   0.873     0.869   61.000
weighted avg      0.877   0.869     0.869   61.000


In [8]:
confusion = pd.crosstab(
    pd.Series(y_test, name="actual"),
    pd.Series(logistic_predictions, name="predicted"),
)
print(confusion.to_string())

predicted  0  1
actual         
0          3  1
1          2  0


## Evaluation Figures

In [9]:
plot_confusion_matrix(
    y_test,
    logistic_predictions,
    FIGURES_DIR / "baseline_confusion_matrix.png",
)
plot_roc_curve(
    y_test,
    logistic_probabilities,
    FIGURES_DIR / "baseline_roc_curve.png",
)
plot_precision_recall_curve(
    y_test,
    logistic_probabilities,
    FIGURES_DIR / "baseline_precision_recall_curve.png",
)

## Logistic Regression Coefficients

In [10]:
preprocessor = logistic_model.named_steps["preprocessor"]
classifier = logistic_model.named_steps["classifier"]
coefficient_table = pd.DataFrame(
    {
        "feature": preprocessor.get_feature_names_out(),
        "coefficient": classifier.coef_[0],
    }
)
coefficient_table = coefficient_table.sort_values("coefficient")
plot_logistic_coefficients(
    coefficient_table,
    FIGURES_DIR / "baseline_logistic_coefficients.png",
)
print(coefficient_table.tail(10).round(3).to_string(index=False))

               feature  coefficient
      numeric__oldpeak        0.245
categorical__exang_1.0        0.270
     numeric__trestbps        0.343
   categorical__ca_1.0        0.347
   categorical__ca_3.0        0.508
categorical__slope_2.0        0.606
   categorical__ca_2.0        0.651
  categorical__sex_1.0        0.716
 categorical__thal_7.0        0.826
   categorical__cp_4.0        1.036


## Cautious Interpretation

The logistic regression baseline performs better than the majority-class dummy baseline on this held-out split, but this is not evidence of clinical validity. The coefficients are useful for inspecting model behavior, not for making causal or diagnostic statements. Performance may vary with split strategy, preprocessing choices, and external datasets.

## Limitations

- This is a single historical dataset and a single train/test split.
- The target is a dataset label and is not a substitute for clinical diagnosis.
- Missing values are imputed within the training pipeline, but imputation choices remain simple.
- Encoded categorical variables need careful documentation before interpretation.
- Calibration is assessed separately in `03_model_validation_and_calibration.ipynb`; external validation is not included.

## Next Steps

1. Review leakage assumptions and preprocessing decisions.
2. Add cross-validation and calibration assessment in a later phase.
3. Compare simple interpretable alternatives only after documenting the baseline.
4. Keep all interpretation educational and dataset-specific.